In [2]:
import polars as pl
from pathlib import Path

carpeta = Path(r"C:\Users\anali\bases_comerciales\projects\bendo\extraer_bendo\backups")

dfs = {
    f.stem: pl.read_parquet(f)
    for f in carpeta.glob("*.parquet")
}

In [5]:
contactabilidad = dfs['contactabilidad_0']

In [7]:
#DF DE INFOR GENERAL
df_info_general = dfs['informacion_general_0']

In [8]:
#DFS de facturas
dfs_facturas = {nombre: df for nombre, df in dfs.items() if "fact" in nombre}

In [9]:
#DFS de balances
dfs_balances = {nombre:df for nombre, df in dfs.items() if "balance" in nombre}

In [10]:
dfs_balances['balances_0'] = (
    dfs_balances['balances_0']
    .drop('anio', 'rama_actividad', 'ciiu', 'cuenta_numero', 'expediente', 'cuenta')
    .rename({'valor': 'valor_balance_2024', 'ruc': 'numero_ruc'})
)

In [11]:
#concatenar Facturas
consolidado_facturas = pl.concat([df_facturas for df_facturas in dfs_facturas.values()])

In [12]:
consolidado_facturas = consolidado_facturas.rename({'codigo_establecimiento': 'numero_establecimiento'})

In [13]:
df_info_general = df_info_general.with_columns(
    pl.col("numero_ruc").cast(pl.Int64)
)

consolidado_facturas = consolidado_facturas.with_columns(
    pl.col("numero_ruc").cast(pl.Int64)
)

contactabilidad = contactabilidad.with_columns(
    pl.col("numero_ruc").cast(pl.Int64)
)

#Añadimos Facturación

df_info_general_facturas = df_info_general.join(
    consolidado_facturas, on=["numero_ruc", "numero_establecimiento"], how="left"
)

#Añadimos balances

df_balaneces_facturas_balance = df_info_general_facturas.join(
    dfs_balances['balances_0'], on="numero_ruc", how="left"
)

#Añadimos contactabilidad

df_final = df_balaneces_facturas_balance.join(
    contactabilidad, on = 'numero_ruc', how = 'left'
)